# Laborator 3 · Clustering (învățare nesupervizată)
**Dataset:** Online Retail II (UCI) · **Metodologie:** CRISP-DM

| Fază CRISP-DM | Secțiune |
|---|---|
| 1. Înțelegerea sarcinii | 0 |
| 2. Înțelegerea datelor | 2 |
| 3. Pregătirea datelor | 3 |
| 4. Modelare | 4 (K-means, ierarhic, DBSCAN, GMM) |
| 5. Evaluare | 5 |
| 6. Implementare | 6 (+ pagina Streamlit) |

## 0. Înțelegerea sarcinii (Business understanding)
**Întrebare:** există grupuri naturale de clienți, după *cum cumpără* (recență, frecvență, valoare)? Segmentarea clienților pentru marketing direcționat este aplicația tipică a clustering-ului.

**Tip de problemă:** nesupervizată, deci nu avem o țintă. Nu putem calcula „acuratețea”, așa că succesul se judecă pe trei criterii:
1. **Separare internă:** silhouette mare, Davies-Bouldin mic, Calinski-Harabasz mare.
2. **Interpretabilitate:** fiecare segment se poate descrie în cuvinte și are o dimensiune utilă (nu 3 clienți).
3. **Utilitate:** segmentele diferă în comportamentul *viitor* (ultimele 90 de zile), pe care algoritmul nu l-a văzut. Îl folosim ca etichetă externă pentru omogenitate, completitudine și V-measure.

**Date de intrare:** tabelul de clienți din Laboratoarele 1–2 (variabile calculate doar din istoric, înainte de data de tăiere).

## 1. Setup

In [ ]:
import json
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import (calinski_harabasz_score, completeness_score, davies_bouldin_score,
                             homogeneity_score, silhouette_score, v_measure_score)
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from baza_date import (ARTEFACTE, SEED, construieste_clienti, curata, incarca_brut, salveaza_fig)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["PYTHONWARNINGS"] = "ignore"
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.3f}".format)

try:                                   # `display` există în Jupyter/VS Code; în terminal îl înlocuim cu print
    display
except NameError:
    def display(obj):
        print(obj)

## 2. Înțelegerea datelor (Data understanding)
### 2.1 Încărcare și curățare
Folosim aceeași încărcare și aceeași curățare în straturi ca în Laboratoarele 1–2 (cod comun în `baza_date.py`), ca toate laboratoarele să pornească din aceleași date.

In [ ]:
df = incarca_brut()
tranzactii, retururi, administrative, ajustari_stoc, contabilitate = curata(df)
contabilitate.to_csv(ARTEFACTE / "curatare.csv", index=False)
print("Dimensiune brută:", df.shape, "| perioadă:", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())
display(contabilitate)

### 2.2 Tabelul de clienți
Un rând = un client; variabilele vin doar din istoric (înainte de data de tăiere), iar `a_cumparat` (dacă a cumpărat în ultimele 90 de zile) îl păstrăm **doar pentru evaluare**: nu intră în clustering.

In [ ]:
clienti, CUTOFF, istoric_v, viitor_v = construieste_clienti(tranzactii, retururi)
print(f"Data de tăiere: {CUTOFF.date()} | clienți cu istoric: {len(clienti)} | au cumpărat în fereastra viitoare: {clienti['a_cumparat'].mean():.1%}")
display(clienti.describe().T)

### 2.3 Ce variabile descriu „cum cumpără” un client?
Cadrul clasic în segmentarea clienților este **RFM**: **R**ecență (cât de recent a cumpărat), **F**recvență (câte facturi), **M**onetar (cât a cheltuit).
Pornim de la patru candidate: R, F, M total și V = valoarea medie a unei facturi.
**De ce le transformăm?** K-means și clusteringul ierarhic folosesc distanța euclidiană, iar variabilele cu coadă lungă ar domina distanța. Aplicăm `log1p` celor cu asimetrie > 1 (regulă practică pentru „puternic asimetric”). Verificăm cu cifre, nu presupunem.

In [ ]:
BRUT = {"R": "recenta_zile", "F": "frecventa", "M": "vanzari_brute", "V": "valoare_medie_factura"}
asimetrie = pd.DataFrame({
    "variabila": list(BRUT.values()),
    "asimetrie_bruta": [clienti[c].skew() for c in BRUT.values()],
    "asimetrie_log1p": [np.log1p(clienti[c]).skew() for c in BRUT.values()],
}, index=list(BRUT.keys()))
asimetrie["transformare"] = np.where(asimetrie["asimetrie_bruta"] > 1, "log1p", "fără")
asimetrie.to_csv(ARTEFACTE / "lab3_asimetrie.csv")
display(asimetrie)

fig, ax = plt.subplots(2, 4, figsize=(14, 5))
for j, (k, c) in enumerate(BRUT.items()):
    clienti[c].hist(bins=40, ax=ax[0, j]); ax[0, j].set_title(f"{k} · {c}", fontsize=9)
    np.log1p(clienti[c]).hist(bins=40, ax=ax[1, j]); ax[1, j].set_title("log1p", fontsize=9)
plt.tight_layout()
salveaza_fig("lab3_distributii")

### 2.4 Redundanță între candidate
Distanța euclidiană numără de două ori informația repetată. `M` total este (aproape) `F × V` (identitate: vânzări = nr. facturi × valoare medie), iar în Laboratorul 1 am văzut că ea produce multicoliniaritate (VIF mare). Verificăm corelațiile pe scara transformată.

In [ ]:
candidati = pd.DataFrame({k: (np.log1p(clienti[c]) if asimetrie.loc[k, "transformare"] == "log1p" else clienti[c])
                          for k, c in BRUT.items()})       # log1p doar unde tabelul de asimetrie a cerut-o
corelatie_cand = candidati.corr()
corelatie_cand.to_csv(ARTEFACTE / "lab3_corelatie_candidati.csv")
display(corelatie_cand)

## 3. Pregătirea datelor (Data preparation)
**Decizie:** folosim **R, F, V** (recență, frecvență, valoare medie pe factură). `M` total este exact `F × V` (prin definiția lui `V`), deci ar număra de două ori aceeași informație în distanță; corelațiile de mai sus arată cât de legată e `M` de `F` și `V`.
**Scalare:** `StandardScaler`, pentru că distanța euclidiană depinde de unitățile de măsură (zile vs. lire vs. număr de facturi).
**Extreme:** le verificăm cu Z-score și IQR. Nu ștergem clienții cu valori mari: sunt tocmai clienții valoroși pe care segmentarea trebuie să-i găsească, iar ștergerea lor ar elimina segmentul de angrosiști; `log1p` le reduce deja influența.

In [ ]:
CARACT = ["R", "F", "V"]
LOG_PE = [k for k in CARACT if asimetrie.loc[k, "transformare"] == "log1p"]    # le păstrăm pentru aplicația Streamlit
NUME_X = {k: ("log_" if k in LOG_PE else "") + BRUT[k] for k in CARACT}


def construieste_x(c):
    """Matricea de intrare a clusteringului: R, F, V (cu log1p unde e cazul)."""
    return pd.DataFrame({NUME_X[k]: (np.log1p(c[BRUT[k]]) if k in LOG_PE else c[BRUT[k]]) for k in CARACT}, index=c.index)


X = construieste_x(clienti)
scaler = StandardScaler().fit(X)
Z = scaler.transform(X)

z_df = pd.DataFrame(Z, columns=X.columns)
q1, q3 = X.quantile(0.25), X.quantile(0.75)
iqr = q3 - q1
extreme = pd.DataFrame({
    "extreme_Z>3": (z_df.abs() > 3).sum().values,
    "extreme_IQR": ((X < q1 - 1.5 * iqr) | (X > q3 + 1.5 * iqr)).sum().values,
}, index=X.columns)
extreme["% din clienți (IQR)"] = extreme["extreme_IQR"] / len(X) * 100
extreme.to_csv(ARTEFACTE / "lab3_extreme.csv")
print("Variabile folosite:", list(X.columns), "| clienți:", len(X))
display(extreme)

## 4. Modelare (Modeling)
### 4.1 Câte clustere? Metoda cotului și coeficientul Silhouette
Pentru alegerea lui K în K-means folosim două instrumente:
- **Elbow:** WCSS (suma pătratelor distanțelor la centroid) scade mereu cu K; căutăm „cotul”, unde câștigul începe să se aplatizeze.
- **Silhouette:** `S(i) = (b(i) − a(i)) / max(a(i), b(i))`, între −1 și 1; mai mare = punctele sunt mai aproape de propriul cluster decât de cel mai apropiat alt cluster.
Calculăm și Davies-Bouldin (mai mic = mai bine) și Calinski-Harabasz (mai mare = mai bine).
**Regulă de alegere:** K=2 separă de obicei doar „activi” de „inactivi” (informație prea grosieră pentru marketing), deci căutăm segmente în intervalul **3–8** și alegem **silhouette maxim**.

In [ ]:
rez_k = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Z)
    rez_k.append({"K": k, "WCSS": km.inertia_, "silhouette": silhouette_score(Z, km.labels_),
                  "Davies-Bouldin": davies_bouldin_score(Z, km.labels_),
                  "Calinski-Harabasz": calinski_harabasz_score(Z, km.labels_)})
rez_k = pd.DataFrame(rez_k).set_index("K")
rez_k["scădere WCSS %"] = -rez_k["WCSS"].pct_change() * 100
rez_k.to_csv(ARTEFACTE / "lab3_alegere_k.csv")
K_BUN = int(rez_k.loc[3:8, "silhouette"].idxmax())
display(rez_k)
print("K ales (silhouette maxim în 3–8):", K_BUN)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(rez_k.index, rez_k["WCSS"], "o-"); ax[0].axvline(K_BUN, color="k", ls="--")
ax[0].set_xlabel("K"); ax[0].set_ylabel("WCSS"); ax[0].set_title("Metoda cotului")
ax[1].plot(rez_k.index, rez_k["silhouette"], "o-"); ax[1].axvline(K_BUN, color="k", ls="--")
ax[1].set_xlabel("K"); ax[1].set_ylabel("silhouette"); ax[1].set_title("Coeficientul Silhouette")
plt.tight_layout()
salveaza_fig("lab3_alegere_k")

### 4.2 K-means (model centroid, clustering „hard”)
Pașii algoritmului: alegem K → inițializăm K centroizi → atribuim fiecare punct celui mai apropiat centroid → recalculăm centroizii → repetăm până nu mai apar schimbări. `n_init=10` repetă algoritmul din 10 inițializări și păstrează cea mai bună, ca rezultatul să nu depindă de o alegere norocoasă.

In [ ]:
kmeans = KMeans(n_clusters=K_BUN, n_init=10, random_state=SEED).fit(Z)
et_km = kmeans.labels_
print("Dimensiunea clusterelor:", dict(zip(*np.unique(et_km, return_counts=True))))

### 4.3 Clustering ierarhic aglomerativ
Pornim cu fiecare client ca un cluster și îmbinăm mereu cea mai apropiată pereche. „Cea mai apropiată” depinde de **distanța dintre clustere**: `single` = minimul dintre puncte, `complete` = maximul, `average` = media tuturor perechilor, `centroid` = distanța dintre centroizi. Adăugăm și `ward` (minimizează creșterea varianței la îmbinare), care evită clustere „în lanț”.
Pentru fiecare tăiem arborele la K clustere și comparăm cu aceleași metrici. Coloana `cel mai mare cluster %` arată dacă o metodă a format un cluster uriaș și altele minuscule (efect de lanț).

In [ ]:
leg, rez_h, et_h = {}, [], {}
for metoda in ["single", "complete", "average", "centroid", "ward"]:
    leg[metoda] = linkage(Z, method=metoda)
    et = fcluster(leg[metoda], K_BUN, criterion="maxclust")
    et_h[metoda] = et
    dim = pd.Series(et).value_counts(normalize=True)
    rez_h.append({"legătură": metoda, "clustere": len(dim), "cel mai mare cluster %": dim.max() * 100,
                  "silhouette": silhouette_score(Z, et) if len(dim) > 1 else np.nan,
                  "Davies-Bouldin": davies_bouldin_score(Z, et) if len(dim) > 1 else np.nan,
                  "Calinski-Harabasz": calinski_harabasz_score(Z, et) if len(dim) > 1 else np.nan})
rez_h = pd.DataFrame(rez_h).set_index("legătură")
rez_h.to_csv(ARTEFACTE / "lab3_ierarhic.csv")
display(rez_h)

plt.figure(figsize=(11, 4))
dendrogram(leg["ward"], truncate_mode="lastp", p=30, show_leaf_counts=True, no_labels=True)
plt.title("Dendrograma (Ward, ultimele 30 îmbinări)"); plt.ylabel("distanța de îmbinare")
salveaza_fig("lab3_dendrograma")

### 4.4 DBSCAN (model de densitate)
Doi parametri: **ε** (raza vecinătății) și **min_samples**. Puncte de nucleu (≥ min_samples vecini în ε), de frontieră și zgomot (−1, ignorate de cluster).
- `min_samples = 2 · nr. variabile` (regulă practică; aici 6).
- **ε din graficul k-distanță:** sortăm distanța fiecărui punct la al `min_samples`-lea vecin; „cotul” curbei (punctul cel mai îndepărtat de dreapta care unește capetele) separă punctele din zone dense de cele izolate.
Verificăm și valori în jurul lui ε, ca să vedem cât de sensibil e rezultatul.

In [ ]:
MIN_S = 2 * Z.shape[1]
kdist = np.sort(NearestNeighbors(n_neighbors=MIN_S).fit(Z).kneighbors(Z)[0][:, -1])


def cot(y):
    """Punctul curbei cel mai îndepărtat de dreapta dintre primul și ultimul punct (metoda „kneedle” simplificată)."""
    x = np.linspace(0, 1, len(y))
    yn = (y - y.min()) / (y.max() - y.min())
    return int(np.argmax(np.abs(yn - x)))


EPS = float(kdist[cot(kdist)])
rez_db = []
for mult in [0.5, 0.75, 1.0, 1.5, 2.0]:
    et = DBSCAN(eps=EPS * mult, min_samples=MIN_S).fit_predict(Z)
    nucleu = et != -1
    n_cl = len(set(et[nucleu]))
    rez_db.append({"ε": EPS * mult, "× cot": mult, "clustere": n_cl, "zgomot %": (~nucleu).mean() * 100,
                   "silhouette (fără zgomot)": silhouette_score(Z[nucleu], et[nucleu]) if n_cl > 1 else np.nan})
rez_db = pd.DataFrame(rez_db).set_index("× cot")
rez_db.to_csv(ARTEFACTE / "lab3_dbscan.csv")
display(rez_db)

et_db = DBSCAN(eps=EPS, min_samples=MIN_S).fit_predict(Z)
plt.figure(figsize=(6, 3.8))
plt.plot(kdist); plt.axhline(EPS, color="k", ls="--", label=f"ε = {EPS:.2f}")
plt.xlabel("puncte sortate"); plt.ylabel(f"distanța la al {MIN_S}-lea vecin"); plt.title("Graficul k-distanță"); plt.legend()
salveaza_fig("lab3_dbscan_kdist")

### 4.5 Gaussian Mixture Models (model de distribuție, clustering „soft”)
GMM presupune că datele vin din K distribuții gaussiene și estimează parametrii cu algoritmul **EM** (Expectation–Maximization). Fiecare client primește o **probabilitate** pentru fiecare cluster (soft), nu o etichetă unică.
Avantajul lor: clustere **eliptice** (nu doar sferice ca la K-means). Verificăm asta: comparăm forme de covarianță (`spherical` ≈ K-means, `diag`, `full` = elipse libere) cu **BIC** (mai mic = mai bun; penalizează complexitatea) pentru K în 3–8.
**Atenție la frecvență:** are valori întregi, deci un grup de clienți poate avea exact aceeași valoare. O componentă „lipită” de o singură valoare ajunge cu varianță aproape zero, iar verosimilitatea (și deci BIC) devine artificial de bună. De aceea fixăm o varianță minimă, `reg_covar = 0,01` (pe date standardizate), și urmărim varianța minimă a componentelor în tabel: dacă tinde la zero, BIC-ul nu e de încredere.

In [ ]:
REG_COVAR = 0.01
rez_g = []
for cov in ["spherical", "diag", "full"]:
    for k in range(3, 9):
        g = GaussianMixture(n_components=k, covariance_type=cov, n_init=3, random_state=SEED, reg_covar=REG_COVAR).fit(Z)
        var_min = np.diagonal(g.covariances_, axis1=1, axis2=2).min() if cov == "full" else np.min(g.covariances_)
        rez_g.append({"covarianță": cov, "K": k, "BIC": g.bic(Z), "AIC": g.aic(Z), "varianța minimă": var_min})
rez_g = pd.DataFrame(rez_g)
rez_g.to_csv(ARTEFACTE / "lab3_gmm_bic.csv", index=False)
best = rez_g.loc[rez_g["BIC"].idxmin()]
COV_G, K_G = best["covarianță"], int(best["K"])
display(rez_g.pivot(index="K", columns="covarianță", values="BIC"))
print(f"BIC minim: covarianță '{COV_G}', K = {K_G}")

gmm = GaussianMixture(n_components=K_G, covariance_type=COV_G, n_init=3, random_state=SEED, reg_covar=REG_COVAR).fit(Z)
et_g = gmm.predict(Z)
prob = gmm.predict_proba(Z)
ambigui = pd.Series(prob.max(axis=1) < 0.6).groupby(et_g).mean() * 100
print(f"Clienți cu apartenență nesigură (probabilitate maximă < 0,6): {(prob.max(axis=1) < 0.6).mean():.1%}")
display(ambigui.rename("ambigui % per cluster").to_frame())

fig, ax = plt.subplots(figsize=(6, 3.8))
for cov, d in rez_g.groupby("covarianță"):
    ax.plot(d["K"], d["BIC"], "o-", label=cov)
ax.set_xlabel("K"); ax.set_ylabel("BIC"); ax.set_title("GMM: BIC după forma covarianței"); ax.legend()
salveaza_fig("lab3_gmm_bic")

## 5. Evaluare (Evaluation)
### 5.1 Comparația algoritmilor
**Metrici interne** (nu cer etichete): silhouette ↑, Davies-Bouldin ↓, Calinski-Harabasz ↑.
**Metrici externe** (cer o clasificare de referință): folosim `a_cumparat` (comportamentul din următoarele 90 de zile, nevăzut de algoritm). **Omogenitate** = fiecare cluster conține o singură clasă; **completitudine** = clasa e adunată într-un singur cluster; **V-measure** = media lor armonică.
Atenție la interpretare: segmentele sunt făcute după *cum au cumpărat*, nu pentru a prezice viitorul; valori mici nu înseamnă clustering prost, ci că segmentarea nu coincide cu o singură întrebare da/nu. Ne interesează **comparația** între algoritmi și faptul că segmentele *diferă* în viitor (5.3).
Pentru DBSCAN, zgomotul (−1) îl tratăm ca o clasă separată la metricile externe, iar la cele interne îl excludem.

In [ ]:
y_ref = clienti["a_cumparat"].astype(int).to_numpy()


def evalueaza(nume, et, zgomot=None):
    nucleu = np.ones(len(et), dtype=bool) if zgomot is None else (et != zgomot)
    n_cl = len(set(et[nucleu]))
    ok = n_cl > 1
    return {"algoritm": nume, "clustere": n_cl, "zgomot %": (~nucleu).mean() * 100,
            "silhouette": silhouette_score(Z[nucleu], et[nucleu]) if ok else np.nan,
            "Davies-Bouldin": davies_bouldin_score(Z[nucleu], et[nucleu]) if ok else np.nan,
            "Calinski-Harabasz": calinski_harabasz_score(Z[nucleu], et[nucleu]) if ok else np.nan,
            "omogenitate": homogeneity_score(y_ref, et), "completitudine": completeness_score(y_ref, et),
            "V-measure": v_measure_score(y_ref, et)}


comparatie = pd.DataFrame([
    evalueaza(f"K-means (K={K_BUN})", et_km),
    evalueaza(f"Ierarhic Ward (K={K_BUN})", et_h["ward"]),
    evalueaza(f"Ierarhic average (K={K_BUN})", et_h["average"]),
    evalueaza(f"DBSCAN (ε={EPS:.2f})", et_db, zgomot=-1),
    evalueaza(f"GMM {COV_G} (K={K_G})", et_g),
]).set_index("algoritm")
comparatie.to_csv(ARTEFACTE / "lab3_comparatie.csv")
display(comparatie)

### 5.2 Alegerea modelului final
**Regulă:** dintre algoritmii care pot atribui **clienți noi** (K-means și GMM; ierarhicul și DBSCAN din scikit-learn nu au `predict`) alegem pe cel cu **silhouette maxim**. Segmentarea trebuie să poată fi aplicată mâine unui client nou.

In [ ]:
candidati_final = {f"K-means (K={K_BUN})": ("kmeans", kmeans, et_km), f"GMM {COV_G} (K={K_G})": ("gmm", gmm, et_g)}
NUME_FINAL = max(candidati_final, key=lambda n: comparatie.loc[n, "silhouette"])
tip_final, model_final, et_final = candidati_final[NUME_FINAL]
print("Model final:", NUME_FINAL, "| silhouette =", round(comparatie.loc[NUME_FINAL, "silhouette"], 3))

### 5.3 Profilul segmentelor
Un segment trebuie să se poată povesti. Calculăm, pe scara originală (mediane, rezistente la extreme): recența în zile, numărul de facturi, valoarea medie a facturii.
**Numele segmentelor** vin din reguli, nu din intuiție: comparăm mediana segmentului cu terțilele (33% și 67%) valorilor tuturor clienților. Recență sub primul terțil = „recent”, peste al doilea = „inactiv”; la fel pentru frecvență (rar / frecvent) și valoarea medie a facturii (mică / mare). Astfel numele corespund valorilor reale din tabel (în zile, facturi, lire), nu unei scări standardizate.
Adăugăm:
- **% din vânzări** (cât din venitul istoric aduce segmentul),
- **% a cumpărat în viitor** și **cheltuiala netă medie în viitor**: dovada că segmentele diferă și în ce nu a văzut algoritmul.

In [ ]:
c = clienti.assign(segment=et_final)
z_seg = pd.DataFrame(Z, columns=CARACT, index=clienti.index).groupby(et_final).mean()


COLS = ["recenta_zile", "frecventa", "valoare_medie_factura"]
p33, p67 = clienti[COLS].quantile(1 / 3), clienti[COLS].quantile(2 / 3)
mediane_seg = c.groupby("segment")[COLS].median()


def nivel(r, col, jos, mijloc, sus):
    return jos if r[col] <= p33[col] else sus if r[col] > p67[col] else mijloc


def nume_segment(r):
    return " · ".join([nivel(r, "recenta_zile", "recent", "recență intermediară", "inactiv"),
                       nivel(r, "frecventa", "rar", "frecvență medie", "frecvent"),
                       nivel(r, "valoare_medie_factura", "valoare mică", "valoare medie", "valoare mare")])


nume_seg = mediane_seg.apply(nume_segment, axis=1)
dup = nume_seg.duplicated(keep=False)
nume_seg[dup] = nume_seg[dup] + " [" + nume_seg[dup].index.astype(str) + "]"       # două segmente cu același nume se deosebesc prin număr

profil = pd.DataFrame({
    "nume": nume_seg,
    "clienți": c.groupby("segment").size(),
    "% clienți": c.groupby("segment").size() / len(c) * 100,
    "recență (zile)": c.groupby("segment")["recenta_zile"].median(),
    "facturi": c.groupby("segment")["frecventa"].median(),
    "valoare medie factură (£)": c.groupby("segment")["valoare_medie_factura"].median(),
    "% din vânzări": c.groupby("segment")["vanzari_brute"].sum() / c["vanzari_brute"].sum() * 100,
    "% a cumpărat în viitor": c.groupby("segment")["a_cumparat"].mean() * 100,
    "cheltuială viitor medie (£)": c.groupby("segment")["tinta_cheltuiala"].mean(),
}).sort_values("% din vânzări", ascending=False)
profil.index.name = "segment"
profil.to_csv(ARTEFACTE / "lab3_profil_segmente.csv")
z_seg.to_csv(ARTEFACTE / "lab3_centroizi_z.csv")
display(profil)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
perechi = [("R", "F"), ("R", "V"), ("F", "V")]
cmap = plt.get_cmap("tab10")
for a, (p, q) in zip(ax, perechi):
    for s in sorted(set(et_final)):
        m = et_final == s
        a.scatter(Z[m, CARACT.index(p)], Z[m, CARACT.index(q)], s=6, alpha=0.5, color=cmap(s % 10), label=f"segment {s}")
    a.set_xlabel(f"{p} (standardizat)"); a.set_ylabel(f"{q} (standardizat)")
ax[0].legend(markerscale=3, fontsize=8)
plt.suptitle(f"Segmentele ({NUME_FINAL}) în perechi de dimensiuni")
plt.tight_layout()
salveaza_fig("lab3_proiectii")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
im = ax[0].imshow(z_seg.loc[profil.index], cmap="coolwarm", vmin=-2, vmax=2, aspect="auto")
ax[0].set_xticks(range(3)); ax[0].set_xticklabels(CARACT); ax[0].set_yticks(range(len(profil))); ax[0].set_yticklabels(profil.index)
for i in range(len(profil)):
    for j in range(3):
        ax[0].text(j, i, f"{z_seg.loc[profil.index[i], CARACT[j]]:.2f}", ha="center", va="center", fontsize=9)
ax[0].set_title("Centroizi (medii standardizate)"); ax[0].set_ylabel("segment")
plt.colorbar(im, ax=ax[0])
poz = np.arange(len(profil))
ax[1].bar(poz - 0.2, profil["% clienți"], 0.4, label="% clienți")
ax[1].bar(poz + 0.2, profil["% din vânzări"], 0.4, label="% din vânzări")
ax[1].set_xticks(poz); ax[1].set_xticklabels(profil.index); ax[1].set_xlabel("segment"); ax[1].legend()
ax[1].set_title("Cine aduce venitul")
plt.tight_layout()
salveaza_fig("lab3_centroizi")

## 6. Implementare (Deployment)
Salvăm un `Pipeline` (scalare + model) care primește valorile R, F, V ale unui client (cu aceleași transformări) și întoarce segmentul. Aplicația Streamlit îl încarcă din `artefacte/`.

In [ ]:
model_clustering = Pipeline([("scalare", scaler), ("model", model_final)])
joblib.dump(model_clustering, ARTEFACTE / "model_clustering.joblib")

segmente = pd.DataFrame({"segment": et_final, "nume_segment": [profil.loc[s, "nume"] for s in et_final]}, index=clienti.index)
segmente.index.name = "Customer ID"
segmente.to_csv(ARTEFACTE / "lab3_segmente_clienti.csv")

info3 = {
    "model": NUME_FINAL, "k": int(len(profil)), "variabile_x": list(X.columns), "log_pe": LOG_PE, "brut": {k: BRUT[k] for k in CARACT},
    "silhouette": float(comparatie.loc[NUME_FINAL, "silhouette"]), "davies_bouldin": float(comparatie.loc[NUME_FINAL, "Davies-Bouldin"]),
    "calinski_harabasz": float(comparatie.loc[NUME_FINAL, "Calinski-Harabasz"]), "v_measure": float(comparatie.loc[NUME_FINAL, "V-measure"]),
    "nume_segmente": {int(s): profil.loc[s, "nume"] for s in profil.index},
    "mediane": {BRUT[k]: float(clienti[BRUT[k]].median()) for k in CARACT},
    "k_bun_silhouette": K_BUN, "eps_dbscan": EPS, "min_samples_dbscan": MIN_S, "clienti": int(len(clienti)),
    "silhouette_pe_k": {int(k): float(v) for k, v in rez_k["silhouette"].items()},
    "ierarhic_cel_mai_mare_cluster": {m: float(v) for m, v in rez_h["cel mai mare cluster %"].items()},
    "ierarhic_silhouette": {m: float(v) for m, v in rez_h["silhouette"].items()},
    "dbscan": {"clustere": int(comparatie.loc[f"DBSCAN (ε={EPS:.2f})", "clustere"]), "zgomot_pct": float(comparatie.loc[f"DBSCAN (ε={EPS:.2f})", "zgomot %"]),
               "silhouette": float(comparatie.loc[f"DBSCAN (ε={EPS:.2f})", "silhouette"])},
    "gmm": {"covarianta": str(COV_G), "k": int(K_G), "silhouette": float(comparatie.loc[f"GMM {COV_G} (K={K_G})", "silhouette"]),
            "varianta_minima": float(rez_g.loc[(rez_g["covarianță"] == COV_G) & (rez_g["K"] == K_G), "varianța minimă"].iloc[0])},
    "pct_vanzari_top_segment": float(profil["% din vânzări"].iloc[0]), "pct_clienti_top_segment": float(profil["% clienți"].iloc[0]),
}
(ARTEFACTE / "info_lab3.json").write_text(json.dumps(info3, ensure_ascii=False, indent=2), encoding="utf-8")

# test: reîncărcăm și segmentăm 3 clienți-exemplu (cei mai recenți, cei mai frecvenți, cei mai inactivi)
incarcat = joblib.load(ARTEFACTE / "model_clustering.joblib")
ex = pd.concat([clienti.nsmallest(1, "recenta_zile"), clienti.nlargest(1, "frecventa"), clienti.nlargest(1, "recenta_zile")])
rezultat = pd.DataFrame({"segment": incarcat.predict(construieste_x(ex))}, index=ex.index)
rezultat["nume"] = rezultat["segment"].map(info3["nume_segmente"])
display(rezultat.join(ex[["recenta_zile", "frecventa", "valoare_medie_factura"]]))

## 7. Concluzii și limite
- **Ce am obținut:** o segmentare a clienților în funcție de cum cumpără (recență, frecvență, valoare), cu segmente care diferă clar în comportamentul din următoarele 90 de zile, pe care algoritmul nu l-a văzut.
- **Cât de „naturale” sunt grupurile:** silhouette-ul moderat și graficele arată un nor continuu de clienți, fără goluri între grupuri. Segmentele sunt o împărțire utilă, nu grupuri separate de la sine; DBSCAN și GMM nu găsesc structură mai bună.
- **De ce nu ierarhicul (single / average / centroid):** scorurile lor mari sunt înșelătoare, pentru că aproape toți clienții cad într-un singur cluster.
- **Limite:** o singură dată de tăiere (segmentele se schimbă în timp); clienții fără ID nu apar; RFM nu vede *ce* cumpără clientul (asta urmează în Laboratoarele 4–5); K-means presupune clustere compacte.